# Tagging workflow for `KE - MSM.xlsx`

This notebook performs:
1. Relevancy tagging based on `Input Name` presence in `Headline` or `Opening Text` (case-insensitive).
2. Sentiment re-tagging (`New_Sentiment`) using the rules you provided (keyword-based + TextBlob polarity fallback).
3. Displays and saves the final results to `tagged_results.xlsx`.

Columns expected in the sheet `KE - MSM`:
- `Input Name`
- `Headline`
- `Opening Text`
- `Sentiment`

Run the cells in order. If you don't have `textblob` installed, the notebook will install it.

In [ ]:
# Install required libraries (uncomment and run if needed)
!pip install -q textblob openpyxl
try:
    from textblob import TextBlob
    print('textblob is available')
except Exception as e:
    print('If you see errors later, rerun this cell to ensure textblob is installed:', e)


In [8]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from pathlib import Path
import re

def contains_keyword(text, keywords):
    if pd.isna(text):
        return False
    text_low = str(text).lower()
    for kw in keywords:
        if kw in text_low:
            return True
    return False

def get_text_polarity(text):
    if pd.isna(text) or str(text).strip()=="":
        return 0.0
    return TextBlob(str(text)).sentiment.polarity


In [9]:
# Read the Excel file (file must be in the working directory)
file_path = 'KE - MSM.xlsx'
sheet_name = 'KE - MSM'

if not Path(file_path).exists():
    raise FileNotFoundError(f"{file_path} not found in working directory. Please upload it to the same folder.")

df = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')
print('Initial shape:', df.shape)
display(df.head(10))


Initial shape: (2961, 53)


,Date,Headline,URL,Opening Text,Hit Sentence,Source,Influencer,Country,Subregion,Language,...,Comments,Shares,Reactions,Threads,Is Verified,Body,Parent URL,Document Tags,Document ID,Custom Categories
0,30-Sep-2025 11:25PM,KCB bounce back to winning ways with victory o...,https://www.standardmedia.co.ke/sports/footbal...,KCB Bank Football Club on Tuesday bounced back...,... KCB Bank Football Club on Tuesday bounced ...,The Standard,Washington Onyango,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""vwfYaPfmoVOImAcSErQwddlFt4k""",NaN
1,30-Sep-2025 11:00PM,KCB edge Kariobangi Sharks to bounce back to w...,moodys:publicid:EAST_AFR_STND:_Oqqlbxjwi_77IOA...,KCB Bank Football Club bounced back to winning...,... KCB Bank Football Club bounced back to win...,"Standard, The (Nairobi, Kenya)",Washington Onyango,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""_Oqqlbxjwi_77IOAegp-IcojKYw""",NaN
2,30-Sep-2025 10:38PM,Safaricom: Fuliza service experiencing tempora...,moodys:publicid:STAR_KENYA:VuoNSLaJzm3ZHMM_tLK...,"Safaricom's, Fuliza is experiencing a temporar...",... rolled out in January 2019 through a partn...,"Star, The (Kenya)",NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""VuoNSLaJzm3ZHMM_tLK8g57SkXk""",NaN
3,30-Sep-2025 10:38PM,Safaricom: Fuliza service experiencing tempora...,moodys:publicid:STAR_KENYA:VuoNSLaJzm3ZHMM_tLK...,"Safaricom's, Fuliza is experiencing a temporar...",... rolled out in January 2019 through a partn...,"Star, The (Kenya)",NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""VuoNSLaJzm3ZHMM_tLK8g57SkXk""",NaN
4,30-Sep-2025 10:10PM,KATA members to get a 4 bob fuel discount from...,https://hapakenya.com/2025/09/30/kata-members-...,Rubis Energy Kenya has announced a new partner...,... & Group MD of Rubis Energy Kenya Grace Mat...,Hapa Kenya,Samson Nderi,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""87hRXlch6KQznY5OzF55qLBTqZs""",NaN
5,30-Sep-2025 09:48PM,Safaricom’s Fuliza Service Crippled as Automat...,https://kenyainsights.com/safaricoms-fuliza-se...,Safaricom’s track record on service reliabilit...,... January 2019 launch. Born from a partnersh...,Kenya Insights,NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""R6D6rR1q3oR5WhbHG996jM-vfxM""",NaN
6,30-Sep-2025 09:48PM,Safaricom’s Fuliza Service Crippled as Automat...,https://kenyainsights.com/safaricoms-fuliza-se...,Safaricom’s track record on service reliabilit...,... January 2019 launch. Born from a partnersh...,Kenya Insights,NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""R6D6rR1q3oR5WhbHG996jM-vfxM""",NaN
7,30-Sep-2025 09:35PM,AFRICA CONCOURS D’ELEGANCE WON BY SATI GATA- AURA,https://africantopstories.co.ke/2025/09/30/afr...,"The metallic green 1947 MG/TC, of Sati Gata-Au...","... , the Concours had a number of sponsors an...",Africantopstories co.ke,NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""xyBAtbod9jRichVt-vbqGqqgymw""",NaN
8,30-Sep-2025 09:34PM,SATI GATA- AURA 1947 MG TC WINS THE AFRICA CON...,https://kenyancorporates.co.ke/2025/09/30/sati...,"The metallic green 1947 MG/TC, of Sati Gata-Au...","... , the Concours had a number of sponsors an...",Kenyan corporates,NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""G_XBv5x9Ug5MPitfgD7HBTeqDjA""",NaN
9,30-Sep-2025 09:22PM,5 reasons why you should consider the Family B...,https://insiderkenya.com/2025/09/30/5-reasons-...,Whether you are pursuing personal goals or exp...,... financing facility 5 things you need to kn...,Insider Kenya,NaN,Kenya,NaN,English,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""icDtZKT3yoqC3B1CXNnXlUBKcl0""",NaN


In [10]:
import re

# Function to extract the "Hit Sentence" from the Headline or Opening Text
def extract_hit_sentence(row):
    # Combine Headline and Opening Text into one string
    text = str(row.get("Headline", "")).strip() + " " + str(row.get("Opening Text", "")).strip()
    
    # Split the text into sentences
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text)
    
    # Return the first sentence or empty string if none found
    return sentences[0] if sentences else ''

# Function to tag relevancy
def tag_relevancy(row):
    # Get brand, normalize to lowercase and remove extra spaces
    brand = str(row.get("Input Name", "")).strip().lower()
    if brand == "":
        return "Not Relevant"

    # Get headline, opening text, and hit sentence, normalize to lowercase
    headline = str(row.get("Headline", "")).lower()
    opening = str(row.get("Opening Text", "")).lower()
    hit_sentence = str(row.get("Hit Sentence", "")).lower()

    # Check if brand exists in headline, opening text, or hit sentence
    if brand in headline or brand in opening or brand in hit_sentence:
        return "Relevant"

    return "Not Relevant"

# Apply both the relevancy and hit sentence extraction
df['Hit Sentence'] = df.apply(extract_hit_sentence, axis=1)  # Add Hit Sentence column
df.insert(0, 'Relevancy', df.apply(tag_relevancy, axis=1))   # Add Relevancy column

print('Relevancy tagging and Hit Sentence extraction done. Distribution:')
display(df['Relevancy'].value_counts(dropna=False))
display(df.head(10))


Relevancy tagging and Hit Sentence extraction done. Distribution:


Relevancy
Not Relevant    2961
Name: count, dtype: int64

,Relevancy,Date,Headline,URL,Opening Text,Hit Sentence,Source,Influencer,Country,Subregion,...,Comments,Shares,Reactions,Threads,Is Verified,Body,Parent URL,Document Tags,Document ID,Custom Categories
0,Not Relevant,30-Sep-2025 11:25PM,KCB bounce back to winning ways with victory o...,https://www.standardmedia.co.ke/sports/footbal...,KCB Bank Football Club on Tuesday bounced back...,KCB bounce back to winning ways with victory o...,The Standard,Washington Onyango,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""vwfYaPfmoVOImAcSErQwddlFt4k""",NaN
1,Not Relevant,30-Sep-2025 11:00PM,KCB edge Kariobangi Sharks to bounce back to w...,moodys:publicid:EAST_AFR_STND:_Oqqlbxjwi_77IOA...,KCB Bank Football Club bounced back to winning...,KCB edge Kariobangi Sharks to bounce back to w...,"Standard, The (Nairobi, Kenya)",Washington Onyango,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""_Oqqlbxjwi_77IOAegp-IcojKYw""",NaN
2,Not Relevant,30-Sep-2025 10:38PM,Safaricom: Fuliza service experiencing tempora...,moodys:publicid:STAR_KENYA:VuoNSLaJzm3ZHMM_tLK...,"Safaricom's, Fuliza is experiencing a temporar...",Safaricom: Fuliza service experiencing tempora...,"Star, The (Kenya)",NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""VuoNSLaJzm3ZHMM_tLK8g57SkXk""",NaN
3,Not Relevant,30-Sep-2025 10:38PM,Safaricom: Fuliza service experiencing tempora...,moodys:publicid:STAR_KENYA:VuoNSLaJzm3ZHMM_tLK...,"Safaricom's, Fuliza is experiencing a temporar...",Safaricom: Fuliza service experiencing tempora...,"Star, The (Kenya)",NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""VuoNSLaJzm3ZHMM_tLK8g57SkXk""",NaN
4,Not Relevant,30-Sep-2025 10:10PM,KATA members to get a 4 bob fuel discount from...,https://hapakenya.com/2025/09/30/kata-members-...,Rubis Energy Kenya has announced a new partner...,KATA members to get a 4 bob fuel discount from...,Hapa Kenya,Samson Nderi,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""87hRXlch6KQznY5OzF55qLBTqZs""",NaN
5,Not Relevant,30-Sep-2025 09:48PM,Safaricom’s Fuliza Service Crippled as Automat...,https://kenyainsights.com/safaricoms-fuliza-se...,Safaricom’s track record on service reliabilit...,Safaricom’s Fuliza Service Crippled as Automat...,Kenya Insights,NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""R6D6rR1q3oR5WhbHG996jM-vfxM""",NaN
6,Not Relevant,30-Sep-2025 09:48PM,Safaricom’s Fuliza Service Crippled as Automat...,https://kenyainsights.com/safaricoms-fuliza-se...,Safaricom’s track record on service reliabilit...,Safaricom’s Fuliza Service Crippled as Automat...,Kenya Insights,NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""R6D6rR1q3oR5WhbHG996jM-vfxM""",NaN
7,Not Relevant,30-Sep-2025 09:35PM,AFRICA CONCOURS D’ELEGANCE WON BY SATI GATA- AURA,https://africantopstories.co.ke/2025/09/30/afr...,"The metallic green 1947 MG/TC, of Sati Gata-Au...",AFRICA CONCOURS D’ELEGANCE WON BY SATI GATA- A...,Africantopstories co.ke,NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""xyBAtbod9jRichVt-vbqGqqgymw""",NaN
8,Not Relevant,30-Sep-2025 09:34PM,SATI GATA- AURA 1947 MG TC WINS THE AFRICA CON...,https://kenyancorporates.co.ke/2025/09/30/sati...,"The metallic green 1947 MG/TC, of Sati Gata-Au...",SATI GATA- AURA 1947 MG TC WINS THE AFRICA CON...,Kenyan corporates,NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""G_XBv5x9Ug5MPitfgD7HBTeqDjA""",NaN
9,Not Relevant,30-Sep-2025 09:22PM,5 reasons why you should consider the Family B...,https://insiderkenya.com/2025/09/30/5-reasons-...,Whether you are pursuing personal goals or exp...,5 reasons why you should consider the Family B...,Insider Kenya,NaN,Kenya,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""icDtZKT3yoqC3B1CXNnXlUBKcl0""",NaN


In [11]:
# 2) Sentiment re-tagging per your custom rules
promo_keywords = [
    'appoint', 'appointment', 'appointed', 'promot', 'promotion', 'promoted', 'hire', 'hired', 'joins', 'joined',
    'named as', 'named', 'award', 'awarded', 'wins', 'win', 'recognition', 'recognised', 'recognize', 'promotion',
    'partnership', 'partner', 'partners', 'expansion', 'expand', 'growth', 'growth in', 'increase in', 'acquire', 'acquired'
]

stock_drop_keywords = [
    'stock price', 'share price', 'shares fell', 'shares fall', 'shares plunged', 'dipped', 'down by', 'fell by', 'decline', 'declined',
    'down %', 'down pct', 'slumped', 'slump', 'drop in share', 'price drop', 'prices fell'
]

negative_keywords = [
    'scandal', 'fraud', 'accuse', 'allegation', 'investigation', 'fined', 'fine', 'lawsuit', 'sued', 'breach', 'illegal', 'corrupt',
    'critic', 'criticise', 'criticize', 'court', 'penalty', 'losses', 'loss', 'bankruptcy', 'default', 'delay', 'controvers', 'probe',
    'exploit', 'violat', 'misconduct', 'resign amid', 'resignation', 'delist'
]

def retag_sentiment(row):
    orig = str(row.get('Sentiment', '')).strip()
    # Keep positive as-is
    if orig.lower() == 'positive':
        return 'Positive'

    # Combine relevant text for checks
    text = ' '.join([str(row.get('Headline','')), str(row.get('Opening Text',''))]).strip()
    text_low = text.lower()

    # Rule: promotions/appointments (positive impact)
    if contains_keyword(text_low, promo_keywords):
        return 'Positive'

    # Rule: stock price drop -> Neutral (explicit)
    if contains_keyword(text_low, stock_drop_keywords):
        return 'Neutral'

    # If content expresses positive tone toward brand even if orig was Neutral/Negative -> Positive
    polarity = get_text_polarity(text)
    # threshold chosen to reduce false positives; tweak if needed
    if polarity > 0.12:
        return 'Positive'

    # If it's clearly negative via keywords -> Negative
    if contains_keyword(text_low, negative_keywords):
        return 'Negative'

    # If polarity strongly negative -> Negative
    if polarity < -0.12:
        return 'Negative'

    # Default fallback: keep Neutral
    return 'Neutral'

# Apply only for rows whose original sentiment is Neutral or Negative (per your instruction)
df['New_Sentiment'] = df.apply(lambda r: retag_sentiment(r) if str(r.get('Sentiment','')).strip().lower() in ['neutral','negative'] else r.get('Sentiment',''), axis=1)

print('New sentiment distribution:')
display(df['New_Sentiment'].value_counts(dropna=False))


New sentiment distribution:


New_Sentiment
Positive    2283
Neutral      484
Negative     192
Unknown        2
Name: count, dtype: int64

In [12]:
# Show rows where sentiment changed (sample)
changed = df[df['Sentiment'].fillna('').astype(str) != df['New_Sentiment'].fillna('').astype(str)]
print('Number of changed rows:', changed.shape[0])
display(changed[['Input Name','Headline','Opening Text','Sentiment','New_Sentiment']].head(20))

# Provide a quick side-by-side comparison for review
comparison = df[['Input Name','Headline','Sentiment','New_Sentiment','Relevancy']].copy()
display(comparison.head(50))


Number of changed rows: 616


,Input Name,Headline,Opening Text,Sentiment,New_Sentiment
1,KCB Bank [Kenya] (MSM) - V3,KCB edge Kariobangi Sharks to bounce back to w...,KCB Bank Football Club bounced back to winning...,Neutral,Positive
5,KCB Bank [Kenya] (MSM) - V3,Safaricom’s Fuliza Service Crippled as Automat...,Safaricom’s track record on service reliabilit...,Negative,Neutral
6,Commercial Bank of Africa [Kenya] (MSM) - V3,Safaricom’s Fuliza Service Crippled as Automat...,Safaricom’s track record on service reliabilit...,Negative,Neutral
11,Commercial Bank of Africa [Kenya] (MSM) - V3,Safaricom: Fuliza service experiencing tempora...,The company confirmed that it is aware of the ...,Neutral,Positive
12,KCB Bank [Kenya] (MSM) - V3,Safaricom: Fuliza service experiencing tempora...,The company confirmed that it is aware of the ...,Neutral,Positive
15,Standard Chartered [Kenya] (MSM) - V3,"CBK to Set Maximum M-Pesa Charges, Kenyan Firm...","Hello, I’m Annah. Welcome to today’s edition o...",Neutral,Positive
16,Stanbic Bank [Kenya] (MSM) - V3,"CBK to Set Maximum M-Pesa Charges, Kenyan Firm...","Hello, I’m Annah. Welcome to today’s edition o...",Neutral,Positive
18,Commercial Bank of Africa [Kenya] (MSM) - V3,Safaricom Issues Update on Service Disruption ...,Safaricom's Fuliza has experienced a temporary...,Neutral,Positive
19,KCB Bank [Kenya] (MSM) - V3,Safaricom Issues Update on Service Disruption ...,Safaricom's Fuliza has experienced a temporary...,Neutral,Positive
21,Co-operative Bank [Kenya] (MSM) - V3,How disorganised data is costing enterprises m...,"Walk into any boardroom across Nairobi, Mombas...",Negative,Positive


,Input Name,Headline,Sentiment,New_Sentiment,Relevancy
0,KCB Bank [Kenya] (MSM) - V3,KCB bounce back to winning ways with victory o...,Positive,Positive,Not Relevant
1,KCB Bank [Kenya] (MSM) - V3,KCB edge Kariobangi Sharks to bounce back to w...,Neutral,Positive,Not Relevant
2,KCB Bank [Kenya] (MSM) - V3,Safaricom: Fuliza service experiencing tempora...,Neutral,Neutral,Not Relevant
3,Commercial Bank of Africa [Kenya] (MSM) - V3,Safaricom: Fuliza service experiencing tempora...,Neutral,Neutral,Not Relevant
4,ABSA [Kenya] (MSM) - V3,KATA members to get a 4 bob fuel discount from...,Positive,Positive,Not Relevant
5,KCB Bank [Kenya] (MSM) - V3,Safaricom’s Fuliza Service Crippled as Automat...,Negative,Neutral,Not Relevant
6,Commercial Bank of Africa [Kenya] (MSM) - V3,Safaricom’s Fuliza Service Crippled as Automat...,Negative,Neutral,Not Relevant
7,Stanbic Bank [Kenya] (MSM) - V3,AFRICA CONCOURS D’ELEGANCE WON BY SATI GATA- AURA,Positive,Positive,Not Relevant
8,Stanbic Bank [Kenya] (MSM) - V3,SATI GATA- AURA 1947 MG TC WINS THE AFRICA CON...,Positive,Positive,Not Relevant
9,Commercial Bank of Africa [Kenya] (MSM) - V3,5 reasons why you should consider the Family B...,Positive,Positive,Not Relevant


In [13]:
# Save the final tagged results to Excel
out_file = 'tagged_results.xlsx'
df.to_excel(out_file, index=False)
print(f'Saved tagged results to {out_file} in the working directory.')


Saved tagged results to tagged_results.xlsx in the working directory.
